# Chirality Atlas: Extending Active Matter and Pattern Formation Tutorials

**UCSD Vibe Coding Active Matter 2026 Hackathon**

This notebook is a natural continuation of the two tutorial tracks:
- Tutorial 1: Active particle models (ABP, Vicsek)
- Tutorial 2: Gray-Scott reaction-diffusion pattern formation

We add one new ingredient to both models: **chirality** (omega), a left/right bias in motion.

By the end you will:
- Reproduce the tutorial baselines
- Extend them with chirality
- Build phase diagrams
- Interpret the biological meaning

> **Audience**: mixed biology / physics / data science background.
> Cells are kept short. Markdown explains the science before each code block.
> Estimated runtime in Colab: 10-20 minutes for all cells.

---
## Part 0. Setup

Install packages and clone the repository so we can use the simulation modules.
If you are running locally with the repo already installed, skip the next two cells.

In [ ]:
# Install dependencies (Colab or fresh environment)
# Run this cell first, then restart the runtime if Colab asks
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'matplotlib', 'scipy', 'Pillow', 'imageio',
                'streamlit'], check=False)
print('Packages ready')

In [ ]:
# Clone the Chirality Atlas repo (Colab only)
# Skip this cell if you already have the repo on disk
import os
if not os.path.isdir('chirality'):
    os.system('git clone https://github.com/agentjakey/chirality.git')
    print('Repo cloned')
else:
    print('Repo already present')

# Add src/ to path so we can import chirality modules
if 'chirality' not in sys.path and os.path.isdir('chirality/src'):
    sys.path.insert(0, 'chirality/src')
elif os.path.isdir('src'):
    sys.path.insert(0, 'src')

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')   # use non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

# Reproducibility: fix the random seed throughout
SEED = 42
rng_global = np.random.default_rng(SEED)

# Consistent figure style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (8, 5)

print('Imports done. Random seed:', SEED)
print('Note: all simulations use fixed seeds for reproducibility.')

In [ ]:
# Import simulation modules from the repo
# If the import fails, run the setup cells above first
from chirality.particle_sim import simulate_abp, simulate_chiral_abp, simulate_vicsek_chiral
from chirality.particle_metrics import (
    polar_order, swirl_index, boundary_accumulation,
    average_neighbor_count, polar_order_timeseries,
)
from chirality.pattern_sim import (
    simulate_gray_scott, simulate_feed_gradient,
    simulate_obstacle, simulate_chiral_source_gray_scott,
)
from chirality.pattern_metrics import pattern_strength, count_clusters, field_asymmetry
from chirality.phase_sweeps import sweep_noise_vs_chirality, sweep_gray_scott_F_k

print('All chirality modules imported successfully.')

---
## Part 1. The Question

### What is active matter?

Active matter is any system whose units consume energy to move.
Examples: bacterial colonies, migrating cells, bird flocks, vibrated granular particles.
Unlike equilibrium systems, active matter is always out of thermodynamic equilibrium.

The tutorial ABP (Active Brownian Particle) model captures the simplest version:
each particle self-propels at speed v0 and rotates randomly (noise Dr).

### What is pattern formation?

Reaction-diffusion systems can spontaneously organize into spatial patterns
(spots, stripes, labyrinths) from a nearly uniform initial condition.
The Gray-Scott model is one well-studied example.
The pattern type is controlled by feed rate F and kill rate k.

### What is chirality?

Chirality means a system cannot be superimposed on its mirror image.
In the particle model: a chiral particle curves left or right instead of going straight.
In the pattern model: a chiral source injects chemical at a rotating focal point.

**Central question:**
> We will ask whether a small left/right bias (omega) can change collective behavior.
> Can microscopic handedness reshape macroscopic structure?

---
## Part 2. Particle Model Baseline

We start with the standard Active Brownian Particle (ABP) from Tutorial 1.

The update rule for each particle i:
```
theta_i += sqrt(2 * Dr * dt) * noise   # random rotation
x_i += v0 * cos(theta_i) * dt          # self-propulsion
y_i += v0 * sin(theta_i) * dt
```

Parameters:
- `v0`: self-propulsion speed
- `Dr`: rotational diffusion (noise strength)
- `dt`: time step
- `N`: number of particles
- `L`: box size (periodic boundary conditions)

In [ ]:
# Standard ABP -- no chirality
hist_abp = simulate_abp(
    N=150, L=10.0, v0=0.5, Dr=0.5,
    dt=0.01, n_steps=500, seed=SEED,
    boundary_mode='periodic', save_every=500,
)

# 'hist_abp' contains positions, orientations, and time array
print(f'Simulated {hist_abp.N} particles over {hist_abp.times[-1]:.2f} time units')
print(f'Box size: {hist_abp.L} x {hist_abp.L}')
print(f'Snapshots stored: {len(hist_abp.positions)}')

In [ ]:
# Plot the final particle snapshot
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: scatter plot of positions
ax = axes[0]
ax.scatter(hist_abp.positions[-1, :, 0], hist_abp.positions[-1, :, 1],
           s=15, c='#888888', alpha=0.85)
dx = 0.4 * np.cos(hist_abp.thetas[-1])
dy = 0.4 * np.sin(hist_abp.thetas[-1])
ax.quiver(hist_abp.positions[-1, :, 0], hist_abp.positions[-1, :, 1], dx, dy,
          color='black', alpha=0.3, scale=1.0, scale_units='xy', angles='xy')
ax.set_xlim(0, hist_abp.L)
ax.set_ylim(0, hist_abp.L)
ax.set_aspect('equal')
ax.set_title(f'Baseline ABP  (Dr={0.5}, N={hist_abp.N})')
ax.set_xlabel('x')
ax.set_ylabel('y')

# Right: orientation histogram
ax2 = axes[1]
ax2.hist(hist_abp.thetas[-1] % (2*np.pi), bins=30, color='#888888',
         edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Orientation (rad)')
ax2.set_ylabel('Count')
ax2.set_title('Orientation distribution (should be uniform)')

fig.suptitle('Active Brownian Particle Baseline', fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# Compute order metrics for the baseline
thetas_final = hist_abp.thetas[-1]
phi = polar_order(thetas_final)

print('--- Baseline ABP metrics ---')
print(f'Polar order phi = {phi:.4f}')
print(f'  (phi=1 means all aligned, phi~0 means disordered)')
print(f'  Expected: ~0 for high noise (Dr=0.5)')

# Sanity check: orientation should be approximately uniform
mean_cos = np.mean(np.cos(thetas_final))
mean_sin = np.mean(np.sin(thetas_final))
print(f'  mean_cos={mean_cos:.3f}, mean_sin={mean_sin:.3f}  (both near 0 = uniform)')

# Try changing Dr: with Dr=0.01, phi should be ~1 (persistent alignment)
# TRY: change Dr below and see what phi you get
Dr_test = 0.5
hist_test = simulate_abp(N=150, L=10.0, v0=0.5, Dr=Dr_test,
                         dt=0.01, n_steps=500, seed=SEED,
                         boundary_mode='periodic', save_every=500)
print(f'Test: Dr={Dr_test} -> phi={polar_order(hist_test.thetas[-1]):.4f}')
# With Dr=0.01 you should see phi much closer to 1

---
## Part 3. Add Chirality to Particles

The chiral extension adds one term to the orientation update:
```
theta_i += omega_i * dt + sqrt(2 * Dr * dt) * noise
```

- `omega > 0`: particle curves in one direction (right-handed)
- `omega < 0`: particle curves the other way (left-handed)
- `omega = 0`: reduces to standard ABP

For a single chirality, each particle traces a circle of radius `R_c = v0 / |omega|`.

**Question**: What changes when omega is zero?
> Answer: the particle stops circling and just diffuses. The trajectory becomes
> random walks instead of loops. Run the cell below to see the difference.

In [ ]:
# Compare: omega=0 (standard) vs omega=2 (chiral)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, omega_val, label, color in [
    (axes[0], 0.0,  'omega=0 (standard ABP)',    '#888888'),
    (axes[1], 2.0,  'omega=2 (chiral, right)',    '#315C4C'),
]:
    hist = simulate_chiral_abp(
        N=100, L=10.0, v0=0.5, Dr=0.2, omega=omega_val,
        chirality_mode='right', dt=0.01, n_steps=600, seed=SEED,
        boundary_mode='periodic', repulsion=False, save_every=600,
    )
    ax.scatter(hist.positions[-1, :, 0], hist.positions[-1, :, 1],
               s=15, c=color, alpha=0.85)
    dx = 0.35 * np.cos(hist.thetas[-1])
    dy = 0.35 * np.sin(hist.thetas[-1])
    ax.quiver(hist.positions[-1, :, 0], hist.positions[-1, :, 1], dx, dy,
              color='black', alpha=0.3, scale=1.0, scale_units='xy', angles='xy')
    ax.set_xlim(0, hist.L)
    ax.set_ylim(0, hist.L)
    ax.set_aspect('equal')
    ax.set_title(label)
    phi = polar_order(hist.thetas[-1])
    si  = swirl_index(hist.positions[-1], hist.thetas[-1], hist.L)
    ax.set_xlabel(f'phi={phi:.3f}  swirl={si:.3f}')

fig.suptitle('Standard ABP vs Chiral ABP', fontweight='bold')
fig.tight_layout()
plt.show()

print('Note: polar order is low in both cases (no alignment interaction).')
print('But swirl index is nonzero for chiral particles -- a new kind of order.')

In [ ]:
# Trajectory comparison: short trails showing the circling motion
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, omega_val, label in [
    (axes[0], 0.0, 'omega=0 -- random walks'),
    (axes[1], 2.0, 'omega=2 -- circular orbits'),
]:
    hist = simulate_chiral_abp(
        N=50, L=10.0, v0=0.5, Dr=0.1, omega=omega_val,
        chirality_mode='right', dt=0.01, n_steps=400, seed=SEED,
        boundary_mode='periodic', repulsion=False, save_every=5,
    )
    # Draw short trails for each particle
    n_snap = hist.positions.shape[0]
    n_trail = min(20, n_snap)
    for p in range(hist.N):
        xs = hist.positions[-n_trail:, p, 0]
        ys = hist.positions[-n_trail:, p, 1]
        # Skip trails that cross the periodic boundary
        for t in range(len(xs)-1):
            if abs(xs[t+1]-xs[t]) < hist.L/2 and abs(ys[t+1]-ys[t]) < hist.L/2:
                alpha = 0.1 + 0.8*(t/(n_trail-1))
                ax.plot([xs[t], xs[t+1]], [ys[t], ys[t+1]],
                        c='#315C4C', alpha=alpha, lw=0.8)
    ax.scatter(hist.positions[-1, :, 0], hist.positions[-1, :, 1],
               s=12, c='#C15A3A', zorder=5)
    ax.set_xlim(0, hist.L)
    ax.set_ylim(0, hist.L)
    ax.set_aspect('equal')
    ax.set_title(label)
    ax.set_facecolor('#111111')

fig.suptitle('Particle Trajectories: omega=0 vs omega=2', fontweight='bold')
fig.tight_layout()
plt.show()

# TRY: change Dr from 0.1 to 0.5. What happens to the circle radius?
# TRY: change omega from 2.0 to 0.5. Do the circles get larger?  (R = v0/omega)

In [ ]:
# Boundary edge current: chiral particles in a circular trap
hist_bnd = simulate_chiral_abp(
    N=150, L=10.0, v0=0.5, Dr=0.3, omega=3.0,
    chirality_mode='right', dt=0.01, n_steps=800, seed=SEED,
    boundary_mode='circular_trap', repulsion=False, save_every=800,
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_facecolor('#111111')
circle = plt.Circle((hist_bnd.L/2, hist_bnd.L/2), 0.45*hist_bnd.L,
                     fill=False, ec='#C15A3A', lw=2, ls='--')
ax.add_patch(circle)
ax.scatter(hist_bnd.positions[-1, :, 0], hist_bnd.positions[-1, :, 1],
           s=15, c='#315C4C', alpha=0.9)
ax.set_xlim(0, hist_bnd.L)
ax.set_ylim(0, hist_bnd.L)
ax.set_aspect('equal')
ax.set_title('Boundary Edge Current  (omega=3, circular_trap)', color='white')
ax.tick_params(colors='white')
fig.patch.set_facecolor('#1c1c1c')
fig.tight_layout()
plt.show()

si_bnd = swirl_index(hist_bnd.positions[-1], hist_bnd.thetas[-1], hist_bnd.L)
ba_bnd = boundary_accumulation(hist_bnd.positions[-1], hist_bnd.L)
print(f'Swirl index: {si_bnd:.4f}')
print(f'Boundary accumulation: {ba_bnd:.4f}  (fraction near the wall)')
print('Particles accumulate on the circular wall and orbit in one direction.')
print('This is analogous to right-handed bacteria orbiting near a surface.')

---
## Part 4. Vicsek-Style Alignment

The Vicsek model adds a local alignment interaction: each particle aligns
with the average heading of neighbors within radius R, then adds noise.

```
phi_avg = average heading of neighbors
theta_i += phi_avg + eta * Uniform[-pi, pi]
```

The polar order parameter phi = |mean(exp(i*theta))| measures collective alignment.
- phi = 1: all particles point the same direction (flock)
- phi ~ 0: random orientations (gas phase)

**The Vicsek transition**: as noise eta increases, phi drops from ~1 to ~0.
This is one of the most studied nonequilibrium phase transitions.

**Sanity check**: high noise should reduce polar order.

In [ ]:
# Vicsek model: compare high noise vs low noise
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, eta_val, label in [
    (axes[0], 0.1, 'Low noise (eta=0.1) -- flocking'),
    (axes[1], 0.8, 'High noise (eta=0.8) -- disordered'),
]:
    hist = simulate_vicsek_chiral(
        N=200, L=10.0, v0=0.5, R=1.0, eta=eta_val, omega=0.0,
        dt=0.1, n_steps=500, seed=SEED,
        boundary_mode='periodic', save_every=500,
    )
    phi = polar_order(hist.thetas[-1])
    ax.scatter(hist.positions[-1, :, 0], hist.positions[-1, :, 1],
               s=12, c='#888888', alpha=0.85)
    dx = 0.4 * np.cos(hist.thetas[-1])
    dy = 0.4 * np.sin(hist.thetas[-1])
    ax.quiver(hist.positions[-1, :, 0], hist.positions[-1, :, 1], dx, dy,
              color='black', alpha=0.35, scale=1.0, scale_units='xy', angles='xy')
    ax.set_xlim(0, hist.L)
    ax.set_ylim(0, hist.L)
    ax.set_aspect('equal')
    ax.set_title(f'{label}')
    ax.set_xlabel(f'polar order phi = {phi:.3f}')

# Sanity check: high noise must give lower polar order than low noise
phi_low  = polar_order(simulate_vicsek_chiral(N=200, L=10.0, v0=0.5, R=1.0,
    eta=0.1, omega=0.0, dt=0.1, n_steps=500, seed=SEED,
    boundary_mode='periodic', save_every=500).thetas[-1])
phi_high = polar_order(simulate_vicsek_chiral(N=200, L=10.0, v0=0.5, R=1.0,
    eta=0.8, omega=0.0, dt=0.1, n_steps=500, seed=SEED,
    boundary_mode='periodic', save_every=500).thetas[-1])
sanity_ok = phi_low > phi_high
print(f'Sanity check: phi(low noise)={phi_low:.3f} > phi(high noise)={phi_high:.3f}?',
      'PASS' if sanity_ok else 'FAIL')

fig.suptitle('Vicsek Model: Noise Controls Collective Order', fontweight='bold')
fig.tight_layout()
plt.show()

# TRY: add omega=1.0 to see how chirality competes with alignment

In [ ]:
# Polar order over time: watch the transition settle
hist_settle = simulate_vicsek_chiral(
    N=200, L=10.0, v0=0.5, R=1.0, eta=0.15, omega=0.0,
    dt=0.1, n_steps=600, seed=SEED,
    boundary_mode='periodic', save_every=10,
)

phi_series = polar_order_timeseries(hist_settle.thetas)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_settle.times, phi_series, c='#315C4C', lw=2)
ax.axhline(phi_series[-1], c='#C15A3A', ls='--', label=f'final = {phi_series[-1]:.3f}')
ax.set_xlabel('Time')
ax.set_ylabel('Polar order phi')
ax.set_title('Vicsek Polar Order Over Time  (eta=0.15)')
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
plt.show()

print(f'System reaches phi ~ {phi_series[-1]:.3f} at steady state')
print('A plateau means the simulation has reached a steady state.')

---
## Part 5. Particle Phase Diagram

We sweep two parameters simultaneously to build a phase diagram:
- x-axis: chirality omega (0 = no chirality, 4 = strong chirality)
- y-axis: rotational noise Dr (0.1 = low noise, 3.0 = high noise)

At each grid point we run a short simulation and measure polar order and swirl index.

**Note on coarse grids**: We use a 4x4 grid here for speed in Colab.
The qualitative trends are correct but exact boundaries would shift with a finer grid.

**Expected result**:
- Polar order should be low everywhere (ABP has no alignment interaction)
- Swirl index should be positive where omega is large (chirality creates circulation)

In [ ]:
# Phase sweep: noise (Dr) vs chirality (omega)
# 4x4 grid, N=60, 200 steps per point -- runs in ~1 min in Colab
print('Running particle phase sweep... (4x4 grid, ~1-2 min)')

results_p = sweep_noise_vs_chirality(
    noise_values=np.linspace(0.1, 3.0, 4),
    chirality_values=np.linspace(0.0, 4.0, 4),
    N=60, L=10.0, v0=0.5, dt=0.01, n_steps=200,
    seed=SEED, verbose=True,
)

print('Sweep complete.')
print(f'Polar order range: {results_p["polar_order"].min():.3f} to {results_p["polar_order"].max():.3f}')
print(f'Swirl index range: {results_p["swirl_index"].min():.3f} to {results_p["swirl_index"].max():.3f}')

In [ ]:
# Plot the particle phase diagrams
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

Dr_vals    = results_p['noise_values']
omega_vals = results_p['chirality_values']
extent     = [omega_vals[0], omega_vals[-1], Dr_vals[0], Dr_vals[-1]]

im0 = axes[0].imshow(
    results_p['polar_order'], origin='lower', aspect='auto',
    extent=extent, cmap='viridis', vmin=0, vmax=1, interpolation='nearest')
plt.colorbar(im0, ax=axes[0], label='Polar order')
axes[0].set_xlabel('Chirality omega')
axes[0].set_ylabel('Rotational noise Dr')
axes[0].set_title('Polar Order  [4x4, coarse]')

im1 = axes[1].imshow(
    results_p['swirl_index'], origin='lower', aspect='auto',
    extent=extent, cmap='RdBu_r', vmin=-0.3, vmax=0.3, interpolation='nearest')
plt.colorbar(im1, ax=axes[1], label='Swirl index')
axes[1].set_xlabel('Chirality omega')
axes[1].set_ylabel('Rotational noise Dr')
axes[1].set_title('Swirl Index  [4x4, coarse]')

fig.suptitle('Chiral ABP Phase Diagrams', fontweight='bold')
fig.tight_layout()
plt.show()

print('Interpretation:')
print('- Polar order is low everywhere: ABP has no alignment interaction.')
print('- Swirl index grows with omega: chirality creates net circulation.')
print('- These are two different kinds of order -- each needs its own metric.')

---
## Part 6. Pattern Formation Baseline

Now we switch to Tutorial 2: reaction-diffusion pattern formation.

The Gray-Scott model describes two interacting chemicals u (substrate) and v (activator):
```
u_t = Du * laplacian(u) - u*v^2 + F*(1 - u)
v_t = Dv * laplacian(v) + u*v^2 - (F + k)*v
```

**Parameter guide:**
- `F` (feed rate): how fast substrate u is replenished. Higher F = more fuel.
- `k` (kill rate): how fast activator v degrades. Higher k = less v survives.
- `Du`, `Dv`: diffusion coefficients. Du > Dv allows Turing instability (self-organization).
- `dt=1.0`: time step for explicit Euler integration.

**Standard regimes:**
- Spots: F ~ 0.035, k ~ 0.065
- Labyrinths: F ~ 0.04, k ~ 0.06
- Uniform (no pattern): F or k too high

We plot the `v` field (activator). High v = bright color. Low v = dark.

In [ ]:
# Gray-Scott baseline: spots
# 128x128 grid, 4000 steps -- runs in ~30 sec in Colab
print('Running Gray-Scott spots (128x128, 4000 steps)...')

hist_spots = simulate_gray_scott(
    nx=128, ny=128, Du=0.16, Dv=0.08,
    F=0.035, k=0.065,
    dt=1.0, n_steps=4000, seed=SEED,
    save_every=4000, perturbation_size=4, n_seeds=8,
)

strength = pattern_strength(hist_spots.v_final)
n_clust, _ = count_clusters(hist_spots.v_final, threshold=0.1)
print(f'Pattern strength (std v): {strength:.4f}')
print(f'Cluster count:            {n_clust}')
print(f'Mean v:                   {hist_spots.v_final.mean():.4f}')

In [ ]:
# Compare spots vs labyrinth morphology
print('Running Gray-Scott labyrinth (128x128, 3000 steps)...')
hist_lab = simulate_gray_scott(
    nx=128, ny=128, Du=0.16, Dv=0.08,
    F=0.04, k=0.06,
    dt=1.0, n_steps=3000, seed=SEED,
    save_every=3000, perturbation_size=5, n_seeds=6,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, hist, label, F_val, k_val in [
    (axes[0], hist_spots, 'Spots',     0.035, 0.065),
    (axes[1], hist_lab,   'Labyrinth', 0.04,  0.06),
]:
    im = ax.imshow(hist.v_final.T, origin='lower', cmap='YlOrBr',
                   vmin=0, vmax=0.5, aspect='equal')
    plt.colorbar(im, ax=ax, label='v (activator)')
    s = pattern_strength(hist.v_final)
    n, _ = count_clusters(hist.v_final, threshold=0.1)
    ax.set_title(f'{label}  (F={F_val}, k={k_val})')
    ax.set_xlabel(f'strength={s:.3f}  clusters={n}')

fig.suptitle('Gray-Scott Pattern Formation', fontweight='bold')
fig.tight_layout()
plt.show()

print('Observation: spots have many small clusters; labyrinths have fewer, larger ones.')
print('TRY: change F from 0.035 to 0.06. The pattern should disappear (uniform state).')

---
## Part 7. Pattern Phase Diagram

We sweep F and k to map out which parameter combinations produce patterns.

**Expected result**: pattern strength should peak in the spots/labyrinth region
(F ~ 0.02-0.05, k ~ 0.05-0.07) and be near zero outside that region.

We use a 4x4 grid here for Colab speed. The full sweep in the repo uses 6x6.

In [ ]:
# Gray-Scott F vs k phase diagram
# 4x4 grid, 64x64, 1500 steps per point -- ~2-4 min in Colab
print('Running Gray-Scott F-k sweep (4x4 grid, ~2-4 min)...')

results_gs = sweep_gray_scott_F_k(
    F_values=np.linspace(0.01, 0.07, 4),
    k_values=np.linspace(0.04, 0.07, 4),
    nx=64, ny=64, dt=1.0, n_steps=1500,
    seed=SEED, verbose=True,
)

print('Sweep complete.')
print(f'Pattern strength range: {results_gs["pattern_strength"].min():.4f} to {results_gs["pattern_strength"].max():.4f}')

In [ ]:
# Plot the pattern phase diagrams
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

F_vals = results_gs['F_values']
k_vals = results_gs['k_values']
extent = [k_vals[0], k_vals[-1], F_vals[0], F_vals[-1]]

im0 = axes[0].imshow(
    results_gs['pattern_strength'], origin='lower', aspect='auto',
    extent=extent, cmap='viridis', interpolation='nearest')
plt.colorbar(im0, ax=axes[0], label='Pattern strength (std v)')
axes[0].set_xlabel('Kill rate k')
axes[0].set_ylabel('Feed rate F')
axes[0].set_title('Pattern Strength  [4x4, coarse]')

im1 = axes[1].imshow(
    results_gs['n_clusters'], origin='lower', aspect='auto',
    extent=extent, cmap='plasma', interpolation='nearest')
plt.colorbar(im1, ax=axes[1], label='Cluster count')
axes[1].set_xlabel('Kill rate k')
axes[1].set_ylabel('Feed rate F')
axes[1].set_title('Cluster Count  [4x4, coarse]')

fig.suptitle('Gray-Scott Phase Diagrams  (F vs k)', fontweight='bold')
fig.tight_layout()
plt.show()

print('Note: high pattern strength and high cluster count = well-developed spots.')
print('The top-right corner (high F, high k) should be near-uniform (strength ~ 0).')

---
## Part 8. Creative Hacks

Here we extend the tutorial models with three modifications.
Each modification breaks a symmetry of the original model.

### Hack 1: Feed Gradient

Replace the uniform feed rate F with a spatially varying F(x) = F_left + (F_right - F_left) * x/L.
Different parts of the domain fall in different Gray-Scott regimes.

**Interpretation question**: which side shows spots and which shows a uniform state?

In [ ]:
# Hack 1: Feed gradient
print('Running feed gradient (128x128, 3000 steps)...')
hist_fgr = simulate_feed_gradient(
    nx=128, ny=128, Du=0.16, Dv=0.08,
    F_left=0.02, F_right=0.055, k=0.063,
    dt=1.0, n_steps=3000, seed=SEED,
    save_every=3000, perturbation_size=4, n_seeds=6,
)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(hist_fgr.v_final.T, origin='lower', cmap='YlOrBr',
               vmin=0, vmax=0.5, aspect='equal')
plt.colorbar(im, ax=ax, label='v (activator)')
ax.set_title(f'Feed Gradient  (F: {0.02} left -> {0.055} right)')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.tight_layout()
plt.show()

asym = field_asymmetry(hist_fgr.v_final)
print(f'Left-right asymmetry: {asym:.5f}')
print('Interpretation: high F side has more pattern; low F side may be uniform.')
print('Q: which side (left or right) shows spots? Does this match your expectation?')

### Hack 2: Circular Obstacle

We place a circular no-reaction zone in the center of the domain.
The obstacle prevents both u and v from reacting inside it.

**Interpretation question**: how does the obstacle affect the nearby pattern?
Does it create a defect? Does the pattern recover far from the obstacle?

In [ ]:
# Hack 2: Circular obstacle
print('Running circular obstacle (128x128, 4000 steps)...')
hist_obs = simulate_obstacle(
    nx=128, ny=128, Du=0.16, Dv=0.08,
    F=0.035, k=0.065,
    dt=1.0, n_steps=4000, seed=SEED,
    obstacle_cx=0.5, obstacle_cy=0.5, obstacle_r=0.12,
    save_every=4000, perturbation_size=4, n_seeds=8,
)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(hist_obs.v_final.T, origin='lower', cmap='YlOrBr',
               vmin=0, vmax=0.5, aspect='equal')
plt.colorbar(im, ax=ax, label='v (activator)')
# Draw obstacle circle for reference
import matplotlib.patches as mpatches
nx = hist_obs.v_final.shape[0]
circ = mpatches.Circle((nx*0.5, nx*0.5), nx*0.12,
                         fill=False, ec='white', lw=2, ls='--')
ax.add_patch(circ)
ax.set_title('Obstacle Disrupted Pattern  (dashed = obstacle boundary)')
fig.tight_layout()
plt.show()

s_obs = pattern_strength(hist_obs.v_final)
n_obs, _ = count_clusters(hist_obs.v_final, threshold=0.1)
print(f'Pattern strength: {s_obs:.4f}  Clusters: {n_obs}')
print('Q: do you see the obstacle boundary in the pattern? What shape is the defect?')

### Hack 3: Chiral Source (Toy Model)

We add a rotating injection point that orbits the box center at angular speed source_omega.
At each time step, a small amount of v is injected at the focal point.

The handedness of the rotation introduces a left-right asymmetry.

**Important caveat**: this is a toy model. We are not claiming this represents
any specific biological mechanism. It is a proof of concept that a rotating
symmetry-breaking source can propagate to macroscopic pattern asymmetry.

**Interpretation question**: does the rotating source break left-right symmetry?
Is the effect large or small?

In [ ]:
# Hack 3: Chiral source
print('Running chiral source (128x128, 4000 steps)...')
hist_chi = simulate_chiral_source_gray_scott(
    nx=128, ny=128, Du=0.16, Dv=0.08,
    F=0.035, k=0.065,
    dt=1.0, n_steps=4000, seed=SEED,
    source_strength=0.02, source_omega=0.1,
    source_r_orbit=0.2, source_sigma=0.05,
    save_every=4000, perturbation_size=5, n_seeds=None,
)

# Compare with no-source baseline
print('Running no-source control (same parameters, omega=0)...')
hist_ctrl = simulate_chiral_source_gray_scott(
    nx=128, ny=128, Du=0.16, Dv=0.08,
    F=0.035, k=0.065,
    dt=1.0, n_steps=4000, seed=SEED,
    source_strength=0.0, source_omega=0.0,
    source_r_orbit=0.2, source_sigma=0.05,
    save_every=4000, perturbation_size=5, n_seeds=None,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, hist, label in [
    (axes[0], hist_ctrl, 'No source (omega=0) -- control'),
    (axes[1], hist_chi,  'Chiral source (omega=0.1)'),
]:
    im = ax.imshow(hist.v_final.T, origin='lower', cmap='YlOrBr',
                   vmin=0, vmax=0.5, aspect='equal')
    plt.colorbar(im, ax=ax, label='v')
    asym = field_asymmetry(hist.v_final)
    ax.set_title(label)
    ax.set_xlabel(f'L-R asymmetry = {asym:.5f}')
fig.suptitle('Chiral Source: Does Rotation Break L-R Symmetry?', fontweight='bold')
fig.tight_layout()
plt.show()

asym_ctrl = field_asymmetry(hist_ctrl.v_final)
asym_chi  = field_asymmetry(hist_chi.v_final)
print(f'Asymmetry without source: {asym_ctrl:.5f}')
print(f'Asymmetry with source:    {asym_chi:.5f}')
print(f'Change:                   {abs(asym_chi) - abs(asym_ctrl):.5f}')
print('Note: the effect is small (~0.002). Reproducible with fixed seeds.')
print('TRY: increase source_omega to 0.5. Does the asymmetry grow?')

---
## Part 9. Bridge: Particles and Fields

The two models are very different mathematically.
But they share a common conceptual structure:

| | Particle model | Field model |
| --- | --- | --- |
| Baseline | ABP, Vicsek | Gray-Scott spots |
| Order parameter | Polar order phi | Pattern strength std(v) |
| Chirality hack | omega*dt in orientation | Rotating source |
| Effect of chirality | Swirl, edge current | L-R asymmetry |
| Control knob | Dr vs omega | F vs k vs omega |

> **The shared idea**: local rules and broken symmetry create global structure.
> In both models, adding a small left/right bias (omega) creates new collective behavior
> that is invisible to the original order parameters.

In [ ]:
# Side-by-side summary: particle chirality vs field chirality
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: boundary edge current (particle track)
ax = axes[0]
ax.set_facecolor('#111111')
circle = plt.Circle((hist_bnd.L/2, hist_bnd.L/2), 0.45*hist_bnd.L,
                     fill=False, ec='#C15A3A', lw=2, ls='--')
ax.add_patch(circle)
ax.scatter(hist_bnd.positions[-1, :, 0], hist_bnd.positions[-1, :, 1],
           s=12, c='#315C4C', alpha=0.9)
ax.set_xlim(0, hist_bnd.L)
ax.set_ylim(0, hist_bnd.L)
ax.set_aspect('equal')
ax.set_title('Particle track: edge current (omega=3)', color='white')
ax.set_xlabel('Swirl and boundary accumulation emerge', color='white')
ax.tick_params(colors='white')

# Right: chiral source pattern (field track)
ax2 = axes[1]
im = ax2.imshow(hist_chi.v_final.T, origin='lower', cmap='YlOrBr',
                vmin=0, vmax=0.5, aspect='equal')
plt.colorbar(im, ax=ax2, label='v')
ax2.set_title('Pattern track: chiral source (omega=0.1)')
ax2.set_xlabel(f'L-R asymmetry = {field_asymmetry(hist_chi.v_final):.5f}')

fig.suptitle('Bridge: local chirality -> global structure (in two models)',
             fontweight='bold')
fig.tight_layout()
plt.show()

print('Both models show the same principle:')
print('  A small left/right bias at the local level reshapes collective behavior.')
print('  The metric that matters is different in each case.')

---
## Part 10. Mini-Challenge Template

Use this template to describe a modification you tried:

```
I changed: [what parameter or model element you modified]

I measured: [what metric you used]

I found: [what the result was, with a number if possible]

This suggests: [one-sentence scientific interpretation]

The model ignores: [one real biological factor this model leaves out]
```

**Example:**
```
I changed: omega from 0 to 3.0 in the circular_trap boundary mode.

I measured: boundary_accumulation fraction.

I found: boundary accumulation went from ~0.10 (omega=0) to ~0.35 (omega=3).

This suggests: chiral particles are pushed to boundaries by their circular
  motion, similar to the way right-handed bacteria orbit near a flat surface.

The model ignores: hydrodynamic interactions between particles,
  and the actual shape/flagella structure that determines omega.
```

In [ ]:
# Try your own modification here
# Replace the values below and run the cell

# --- YOUR MODIFICATION ---
my_omega = 1.5      # try 0.0, 1.0, 2.0, 4.0
my_Dr    = 0.3      # try 0.1 (low noise) or 2.0 (high noise)
my_mode  = 'right'  # try 'left', 'right', 'racemic', 'random'
# --- END MODIFICATION ---

hist_mine = simulate_chiral_abp(
    N=150, L=10.0, v0=0.5, Dr=my_Dr, omega=my_omega,
    chirality_mode=my_mode, dt=0.01, n_steps=600, seed=SEED,
    boundary_mode='circular_trap', repulsion=False, save_every=600,
)

phi_mine = polar_order(hist_mine.thetas[-1])
si_mine  = swirl_index(hist_mine.positions[-1], hist_mine.thetas[-1], hist_mine.L)
ba_mine  = boundary_accumulation(hist_mine.positions[-1], hist_mine.L)

print(f'omega={my_omega}, Dr={my_Dr}, mode={my_mode}')
print(f'  polar order:           {phi_mine:.4f}')
print(f'  swirl index:           {si_mine:.4f}')
print(f'  boundary accumulation: {ba_mine:.4f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_facecolor('#111111')
circle = plt.Circle((hist_mine.L/2, hist_mine.L/2), 0.45*hist_mine.L,
                     fill=False, ec='#C15A3A', lw=1.5, ls='--')
ax.add_patch(circle)
om = hist_mine.omegas
om_max = max(abs(om).max(), 1e-9)
colors = plt.cm.RdBu(mcolors.Normalize(-om_max, om_max)(om))
ax.scatter(hist_mine.positions[-1, :, 0], hist_mine.positions[-1, :, 1],
           c=colors, s=15, alpha=0.9)
ax.set_xlim(0, hist_mine.L)
ax.set_ylim(0, hist_mine.L)
ax.set_aspect('equal')
ax.set_title(f'omega={my_omega}, Dr={my_Dr}, {my_mode}', color='white')
ax.tick_params(colors='white')
fig.patch.set_facecolor('#1c1c1c')
fig.tight_layout()
plt.show()

---
## Part 11. Hackathon Slide Checklist

Before presenting, check that your 5-slide deck covers:

**Slide 1: Question and motivation**
- [ ] Central question stated in one sentence
- [ ] At least one biological example of chirality
- [ ] Connection to the tutorial models named explicitly

**Slide 2: Model**
- [ ] Equations or pseudocode for your model extension
- [ ] Which tutorial it extends and how
- [ ] What parameter you added or changed

**Slide 3: Baseline result**
- [ ] Baseline simulation image (ABP or Gray-Scott without chirality)
- [ ] At least one metric reported with a number
- [ ] Sanity check: does it match the tutorial expectation?

**Slide 4: Creative modification**
- [ ] Image showing the effect of your modification
- [ ] Metric comparison: with vs without your modification
- [ ] Phase diagram or at least a 2-point comparison

**Slide 5: Biological interpretation and limitations**
- [ ] One specific biological analog named
- [ ] At least one honest limitation stated
- [ ] Takeaway sentence: what does this suggest more broadly?

---

**A note on honesty in science:**
The most compelling hackathon submissions are ones that make a modest,
defensible claim and state clearly what the model cannot do.
A small, real effect is more interesting than a large, overclaimed one.

In [ ]:
# Final sanity check: run smoke test on the modules used in this notebook
import importlib

modules = [
    'chirality.particle_sim',
    'chirality.particle_metrics',
    'chirality.pattern_sim',
    'chirality.pattern_metrics',
    'chirality.phase_sweeps',
]

all_ok = True
for mod in modules:
    try:
        importlib.import_module(mod)
        print(f'  OK   {mod}')
    except Exception as e:
        print(f'  FAIL {mod}: {e}')
        all_ok = False

print()
print('Notebook complete.' if all_ok else 'Some modules failed to import.')
print('Estimated total runtime: 10-20 minutes with all cells run top-to-bottom.')